📦 Bloque 0 — Imports estándar + módulo `functions` + carga config JSON

In [ ]:
from pathlib import Path
import sys
import importlib
import json
import os
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config" / "CCLA"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_monitoreo.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))

except FileNotFoundError:
    raise FileNotFoundError(f"No se encontró: {config_file}")
except json.JSONDecodeError as e:
    raise ValueError(f"JSON inválido en {config_file}: {e}")

# --- Extraer shortcuts desde config ---
T = data["tablas_remotas"]
P = data["periodo"]

oficio_suffix      = P["oficio_suffix"]
corte_general      = P["corte_general"]
corte_especial     = P["corte_especial"]
vigencia_general   = P["vigencia_general"]
vigencia_especial  = P["vigencia_especial"]
polizas_especiales = P["polizas_especiales"]

# Nombre dinámico de tabla oficio
oficio_table = f"Oficio_LA_Araucana_PPI{oficio_suffix}_BKP"

# Polizas como string SQL
polizas_sql = ", ".join(str(p) for p in polizas_especiales)

print(f"\n📅 Período configurado:")
print(f"   Oficio suffix     : {oficio_suffix}")
print(f"   Corte general     : {corte_general}")
print(f"   Corte especial    : {corte_especial}")
print(f"   Vigencia general  : {vigencia_general}")
print(f"   Vigencia especial : {vigencia_especial}")
print(f"   Pólizas especiales: {polizas_especiales}")
print(f"   Tabla oficio      : {oficio_table}")

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
FUNCTIONS_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\functions exists: True
Importado desde: C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py


📦 Bloque 1 — Conexión SQL Server

In [ ]:
# --- Parámetros desde config ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

driver = "ODBC Driver 17 for SQL Server"

# 1) ODBC connection string (pyodbc)
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# 2) SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",
)

## CONSOLIDACIÓN: TOTAL_ARAUCANA

📦 Bloque 2 — DROP TOTAL_ARAUCANA_BKP

In [ ]:
query_monitoreo_1 = f"""
DROP TABLE IF EXISTS {T["total_araucana"]};
"""

fun.exec_sql(query_monitoreo_1, connection_string)

📦 Bloque 3 — SELECT INTO TOTAL_ARAUCANA_BKP (UNION ALL)

In [ ]:
query_monitoreo_2 = f"""
SELECT T.*, CONVERT(FLOAT,'0') AS Comision
INTO {T["total_araucana"]}
FROM (
    -- [MARSH]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           Prima_Bruta_mensual AS PrimaBruta, Prima_Neta AS PrimaNeta
    FROM {T["marsh_final"]}

    UNION ALL

    -- [CONOSUR]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBrutaMensual AS PrimaBruta, PrimaNetaMensual AS PrimaNeta
    FROM {T["conosur_final"]}

    UNION ALL

    -- [VOLVEK_FLUJO2]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM {T["flujo2_final"]}

    UNION ALL

    -- [VOLVEK_FLUJO]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM {T["flujo_final"]}

    UNION ALL

    -- [VOLVEK_STOCK]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM {T["stock_final"]}

    UNION ALL

    -- [VOLVEK_PLANES]
    SELECT MES_Recaudacion, POLIZA, foliocredito, rutafiliado, MontoBruto, Plazo,
           fechaPrimerVto, FechaUltimoVto, ValorCuota, NULL AS FechaPrima, Producto,
           FolioOrigen, FechaOrigen, PLAN_TECNICO, PLAZO_CUOTAS, Negocio,
           PrimaBruta, PrimaNeta
    FROM {T["planes_final"]}
) AS T;
"""

fun.exec_sql(query_monitoreo_2, connection_string)

## COMISIONES: UPDATE SOBRE TOTAL_ARAUCANA

📦 Bloque 4 — UPDATE comisiones por Plan Técnico

In [ ]:
query_monitoreo_3 = f"""
UPDATE {T["total_araucana"]}
SET Comision = CASE
    WHEN Plan_Tecnico = 4277 THEN 0.0476 * PrimaNeta
    WHEN Plan_Tecnico IN (4331, 4332, 4333, 4334, 4422) THEN 0.0874 * PrimaNeta
    WHEN Plan_Tecnico = 6270 THEN 0.0665 * PrimaNeta
    WHEN Plan_Tecnico IN (6832, 8285) THEN 0.0087 * PrimaNeta
    ELSE Comision -- mantiene
END;

PRINT('fin de la base manual');
"""

fun.exec_sql(query_monitoreo_3, connection_string)

📦 Bloque 5 — Validación rápida consolidado

In [ ]:
query_monitoreo_4 = f"""
-- Validación rápida consolidado
SELECT TOP 20 *
FROM {T["total_araucana"]};
"""

df_validacion = fun.query_to_df(
    sql=query_monitoreo_4,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_validacion)

## MONITORING: CREACIÓN TABLA DESTINO

📦 Bloque 6 — DROP Monitoring_LaAraucana_BKP

In [ ]:
query_monitoreo_5 = f"""
DROP TABLE IF EXISTS {T["monitoring"]};
"""

fun.exec_sql(query_monitoreo_5, connection_string)

📦 Bloque 7 — CREATE TABLE Monitoring_LaAraucana_BKP

In [ ]:
query_monitoreo_6 = f"""
CREATE TABLE {T["monitoring"]}
(
   NANOPROC NUMERIC(4,0),
   NMESPROC NUMERIC(2,0),
   CCODLNRO VARCHAR(11),
   LN NUMERIC(9,0),
   NNUMDOCU NUMERIC(20,0),
   NNUMENDO BIGINT,
   NNUMITEM BIGINT,
   NNUMCERT BIGINT,
   PROPUESTA BIGINT,
   CCODCOBE VARCHAR(5),
   CCODRAMO VARCHAR(2),
   NCORASVS NUMERIC(5,0),
   PLAN_TECNICO NUMERIC(5,0),
   RAMO_IBNR VARCHAR(50),
   PARTNER_SUCURSAL VARCHAR(100),
   BU VARCHAR(70),
   COBERTURA1 VARCHAR(50),
   COBERTURA2 VARCHAR(80),
   TIPO_COBERTURA CHAR(40),
   CONTRATO_REAS CHAR(50),
   MCA_VIG VARCHAR(2),
   TIPO VARCHAR(6),
   RUT_INT NUMERIC(10,0),
   PARTNER_PYG NVARCHAR(255),
   PLAN_COMERCIAL_GOL NUMERIC(5,0),
   CANAL_GOL NVARCHAR(35),
   SUCURSAL_GOL VARCHAR(60),
   RUT_INTERMEDIARIO_GOL NUMERIC(10,0),
   RUT_EJECUTIVO_GOL NUMERIC(10,0),
   PARTNER_GOL VARCHAR(100),
   BU_PYG NVARCHAR(50),
   ZONA_SUCURSAL NVARCHAR(50),
   RAMO_IBNR_ORIG NVARCHAR(50),
   COBERTURA_AGRUP NVARCHAR(50),
   NFECEMIS DATE,
   NFEINVIG DATE,
   NFETEVIG DATE,
   CCOMOORI VARCHAR(3),
   TC INT,
   TC_UF FLOAT,
   PRIMA FLOAT,
   COMISION FLOAT,
   PRIMA_CLP FLOAT,
   COMISION_CLP FLOAT,
   PRIMA_UF FLOAT,
   COMISION_UF FLOAT
);
"""

fun.exec_sql(query_monitoreo_6, connection_string)

## MONITORING: INSERT DETALLE (por cobertura real desde EMISION_DEVENGADA_PPI)

📦 Bloque 8 — INSERT detalle coberturas

In [ ]:
query_monitoreo_7 = f"""
INSERT INTO {T["monitoring"]}
SELECT
    YEAR(f.PER_ANT) AS NANOPROC,
    MONTH(f.PER_ANT) AS NMESPROC,
    a.CCODLNRO,
    a.LN,
    b.POLIZA AS NNUMDOCU,
    0 AS NNUMENDO,
    CONVERT(BIGINT, b.foliocredito) AS NNUMITEM,
    b.rutafiliado AS NNUMCERT,
    CONVERT(BIGINT, b.foliocredito) AS PROPUESTA,
    a.CCODCOBE,
    a.CCODRAMO,
    a.NCORASVS,
    a.PLAN_TECNICO,
    a.RAMO_IBNR,
    a.PARTNER_SUCURSAL,
    a.BU,
    a.COBERTURA1,
    a.COBERTURA2,
    a.TIPO_COBERTURA,
    a.CONTRATO_REAS,
    a.MCA_VIG,
    a.TIPO,
    a.RUT_INT,
    a.PARTNER_PYG,
    a.PLAN_COMERCIAL_GOL,
    a.CANAL_GOL,
    a.SUCURSAL_GOL,
    a.RUT_INTERMEDIARIO_GOL,
    a.RUT_EJECUTIVO_GOL,
    a.PARTNER_GOL,
    a.BU_PYG,
    a.ZONA_SUCURSAL,
    a.RAMO_IBNR_ORIG,
    a.COBERTURA_AGRUP,
    f.PER AS NFECEMIS,
    f.PER_ANT AS NFEINVIG,
    DATEADD(day, 1, f.PER) AS NFETEVIG,
    'CLP' AS CCOMOORI,
    0 AS TC,
    c.TC_UF,
    b.PrimaNeta AS PRIMA,
    b.Comision * -1 AS COMISION,
    b.PrimaNeta AS PRIMA_CLP,
    b.Comision * -1 AS COMISION_CLP,
    (b.PrimaNeta / c.TC_UF) AS PRIMA_UF,
    (b.Comision * -1 / c.TC_UF) AS COMISION_UF
FROM {T["total_araucana"]} b
LEFT JOIN (
    SELECT
        NNUMDOCU,
        CCODLNRO,
        LN,
        CCODCOBE,
        CCODRAMO,
        NCORASVS,
        PLAN_TECNICO,
        RAMO_IBNR,
        PARTNER_SUCURSAL,
        BU,
        COBERTURA1,
        COBERTURA2,
        TIPO_COBERTURA,
        CONTRATO_REAS,
        MCA_VIG,
        CCOMOORI,
        TIPO,
        RUT_INT,
        PARTNER_PYG,
        PLAN_COMERCIAL_GOL,
        CANAL_GOL,
        SUCURSAL_GOL,
        RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL,
        PARTNER_GOL,
        BU_PYG,
        ZONA_SUCURSAL,
        RAMO_IBNR_ORIG,
        COBERTURA_AGRUP,
        count(NNUMDOCU) AS casos
    FROM [Habitat].[dbo].[EMISION_DEVENGADA_PPI]
    WHERE nnumdocu IN (4780715, 4780716, 4780717, 4780718, 4780719, 4659577, 4668266, 5698774, 6354562, 7561166)
    GROUP BY
        NNUMDOCU, CCODLNRO, LN, CCODCOBE, CCODRAMO, NCORASVS, PLAN_TECNICO, RAMO_IBNR, PARTNER_SUCURSAL,
        BU, COBERTURA1, COBERTURA2, TIPO_COBERTURA, CONTRATO_REAS, MCA_VIG, CCOMOORI, TIPO, RUT_INT,
        PARTNER_PYG, PLAN_COMERCIAL_GOL, CANAL_GOL, SUCURSAL_GOL, RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL, PARTNER_GOL, BU_PYG, ZONA_SUCURSAL, RAMO_IBNR_ORIG, COBERTURA_AGRUP
) a
    ON b.POLIZA = a.NNUMDOCU
LEFT JOIN (SELECT * FROM Sura_Periodo) f
    ON b.MES_Recaudacion = f.MES_REF
LEFT JOIN (SELECT *, CAST(fecha as DATE) AS fecha_date FROM TC_DIARIO) c
    ON f.PER = c.fecha_date;
"""

fun.exec_sql(query_monitoreo_7, connection_string)

## (OPCIONAL) CONSULTA SOPORTE UF

📦 Bloque 9 — Consulta TC_DIARIO

In [ ]:
query_monitoreo_8 = """
SELECT TOP 10 *
FROM TC_DIARIO
ORDER BY fecha DESC;
"""

df_tc = fun.query_to_df(
    sql=query_monitoreo_8,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_tc)

## MONITORING: INSERT TOTAL (agrega fila "TOTAL" por póliza)

📦 Bloque 10 — INSERT fila TOTAL desde TOTAL_ARAUCANA (prod)

In [ ]:
query_monitoreo_9 = f"""
INSERT {T["monitoring"]}
SELECT
    YEAR(f.PER_ANT) as NANOPROC,
    MONTH(f.PER_ANT) as NMESPROC,
    a.CCODLNRO,
    a.LN,
    b.POLIZA as NNUMDOCU,
    0 as NNUMENDO,
    CONVERT(BIGINT, b.foliocredito) as NNUMITEM,
    b.rutafiliado as NNUMCERT,
    CONVERT(BIGINT, b.foliocredito) as PROPUESTA,
    1 as CCODCOBE,
    a.CCODRAMO,
    99 as NCORASVS,
    a.PLAN_TECNICO,
    a.RAMO_IBNR,
    a.PARTNER_SUCURSAL,
    a.BU,
    'TOTAL' as COBERTURA1,
    'TOTAL' as COBERTURA2,
    a.TIPO_COBERTURA,
    a.CONTRATO_REAS,
    a.MCA_VIG,
    a.TIPO,
    a.RUT_INT,
    a.PARTNER_PYG,
    a.PLAN_COMERCIAL_GOL,
    a.CANAL_GOL,
    a.SUCURSAL_GOL,
    a.RUT_INTERMEDIARIO_GOL,
    a.RUT_EJECUTIVO_GOL,
    a.PARTNER_GOL,
    a.BU_PYG,
    a.ZONA_SUCURSAL,
    a.RAMO_IBNR_ORIG,
    'TOTAL' as COBERTURA_AGRUP,
    f.PER as NFECEMIS,
    f.PER_ANT as NFEINVIG,
    DATEADD(day, 1, f.PER) AS NFETEVIG,
    'CLP' as CCOMOORI,
    0 as TC,
    c.TC_UF,
    b.PrimaNeta as PRIMA,
    b.Comision as COMISION,
    b.PrimaNeta as PRIMA_CLP,
    b.Comision * -1 as COMISION_CLP,
    (b.PrimaNeta / c.TC_UF) as PRIMA_UF,
    (b.Comision * -1 / c.TC_UF) as COMISION_UF
FROM {T["total_araucana_prod"]} b
LEFT JOIN (
    SELECT
        NNUMDOCU,
        CCODLNRO,
        LN,
        CCODRAMO,
        PLAN_TECNICO,
        RAMO_IBNR,
        PARTNER_SUCURSAL,
        BU,
        TIPO_COBERTURA,
        CONTRATO_REAS,
        MCA_VIG,
        CCOMOORI,
        TIPO,
        RUT_INT,
        PARTNER_PYG,
        PLAN_COMERCIAL_GOL,
        CANAL_GOL,
        SUCURSAL_GOL,
        RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL,
        PARTNER_GOL,
        BU_PYG,
        ZONA_SUCURSAL,
        RAMO_IBNR_ORIG,
        count(NNUMDOCU) as casos
    FROM [Habitat].[dbo].[EMISION_DEVENGADA_PPI]
    WHERE nnumdocu IN (4780715, 4780716, 4780717, 4780718, 4780719, 4659577, 4668266, 5698774, 6354562, 7561166)
    GROUP BY
        NNUMDOCU, CCODLNRO, LN, CCODRAMO, NCORASVS, PLAN_TECNICO, RAMO_IBNR, PARTNER_SUCURSAL,
        BU, TIPO_COBERTURA, CONTRATO_REAS, MCA_VIG, CCOMOORI, TIPO, RUT_INT,
        PARTNER_PYG, PLAN_COMERCIAL_GOL, CANAL_GOL, SUCURSAL_GOL, RUT_INTERMEDIARIO_GOL,
        RUT_EJECUTIVO_GOL, PARTNER_GOL, BU_PYG, ZONA_SUCURSAL, RAMO_IBNR_ORIG
) a
    ON b.POLIZA = a.NNUMDOCU
LEFT JOIN (select * from Sura_Periodo) f
    ON b.MES_Recaudacion = f.MES_REF
LEFT JOIN (select *, CAST(fecha as DATE) as fecha_date from TC_DIARIO) c
    ON f.PER = c.fecha_date;
"""

fun.exec_sql(query_monitoreo_9, connection_string)

## MONITORING: UPDATE FINAL PARTNER

📦 Bloque 11 — UPDATE PARTNER

In [ ]:
query_monitoreo_10 = f"""
UPDATE {T["monitoring"]}
SET PARTNER_SUCURSAL = 'La Araucana',
    PARTNER_GOL      = 'La Araucana';
"""

fun.exec_sql(query_monitoreo_10, connection_string)

# Agregado de suma asegurada desde Cesantía

📦 Bloque 12 — DROP tabla temporal PPI

In [ ]:
query_monitoreo_11 = """
DROP TABLE IF EXISTS #Oficio_LA_Araucana_PPI_BKP;
"""

fun.exec_sql(query_monitoreo_11, connection_string)

📦 Bloque 13 — SELECT INTO #Oficio_LA_Araucana_PPI_BKP (tabla temporal)

In [ ]:
query_monitoreo_12 = f"""
/*--------------------------------------------------------------
 1. CREACIÓN DE TABLA TEMPORAL BASE
    - Extrae información desde Monitoring_LaAraucana
    - Excluye registros agregados (COBERTURA_AGRUP = 'TOTAL')
--------------------------------------------------------------*/

SELECT
    -- Mes de referencia en formato AAAAMM
    (NANOPROC * 100 + NMESPROC) AS mes_referencia,

    -- Tipo de documento asegurado (valor fijo)
    1 AS COD_TIPO_DOCUMENTO_ASEGURADO,

    -- RUT del asegurado
    NNUMCERT AS RUT,

    -- Número de documento / póliza
    NNUMDOCU,

    -- Ítem de la póliza
    NNUMITEM,

    -- Plan técnico del producto
    PLAN_TECNICO,

    -- Código de ramo
    NCORASVS AS COD_RAMO,

    -- Prima expresada en UF
    PRIMA_UF,

    -- Periodicidad de pago (valor fijo)
    3 AS COD_PERIODICIDAD_PAGO,

    -- Tipo de pago (valor fijo)
    9 AS COD_TIPO_PAGO
INTO #Oficio_LA_Araucana_PPI_BKP
FROM {T["monitoring"]}
WHERE COBERTURA_AGRUP != 'TOTAL';
"""

fun.exec_sql(query_monitoreo_12, connection_string)

📦 Bloque 14 — Depuración por mes de corte (parametrizado)

In [ ]:
query_monitoreo_13 = f"""
/*--------------------------------------------------------------
 2. DEPURACIÓN DE REGISTROS POR MES DE CORTE
    - Regla especial para nnumdocu específicos
    - Regla general para el resto de documentos
--------------------------------------------------------------*/

DELETE FROM #Oficio_LA_Araucana_PPI_BKP
WHERE
      nnumdocu IN ({polizas_sql})
      AND mes_referencia < {corte_especial}
   OR nnumdocu NOT IN ({polizas_sql})
      AND mes_referencia < {corte_general};
"""

fun.exec_sql(query_monitoreo_13, connection_string)

📦 Bloque 15 — DROP tabla temporal PPI2

In [ ]:
query_monitoreo_14 = """
DROP TABLE IF EXISTS #Oficio_LA_Araucana_PPI2_BKP;
"""

fun.exec_sql(query_monitoreo_14, connection_string)

📦 Bloque 16 — Consolidación PPI (GROUP BY)

In [ ]:
query_monitoreo_15 = """
/*--------------------------------------------------------------
 3. CONSOLIDACIÓN DE INFORMACIÓN
    - Se agrupa por póliza / ítem / plan
    - Se calcula:
        * Último mes informado (MAX mes_referencia)
        * Prima directa acumulada (SUM PRIMA_UF)
--------------------------------------------------------------*/

SELECT
    -- Último mes informado por póliza/ítem
    MAX(mes_referencia) AS max_mes_referencia,

    COD_TIPO_DOCUMENTO_ASEGURADO,
    RUT,
    NNUMDOCU,
    NNUMITEM,
    PLAN_TECNICO,
    COD_RAMO,

    -- Prima directa acumulada
    SUM(PRIMA_UF) AS PRIMA_DIRECTA,

    COD_PERIODICIDAD_PAGO,
    COD_TIPO_PAGO
INTO #Oficio_LA_Araucana_PPI2_BKP
FROM #Oficio_LA_Araucana_PPI_BKP
GROUP BY
    COD_TIPO_DOCUMENTO_ASEGURADO,
    RUT,
    NNUMDOCU,
    NNUMITEM,
    PLAN_TECNICO,
    COD_RAMO,
    COD_PERIODICIDAD_PAGO,
    COD_TIPO_PAGO;
"""

fun.exec_sql(query_monitoreo_15, connection_string)

📦 Bloque 17 — DROP tabla final oficio

In [ ]:
query_monitoreo_16 = f"""
DROP TABLE IF EXISTS {oficio_table};
"""

fun.exec_sql(query_monitoreo_16, connection_string)

📦 Bloque 18 — SELECT INTO tabla final oficio CMF (parametrizado)

In [ ]:
query_monitoreo_17 = f"""
/*--------------------------------------------------------------
 4. CONSTRUCCIÓN DE TABLA FINAL PARA OFICIO CMF
    - Se agregan datos de la compañía
    - Se determina vigencia según mes de corte
    - Se calcula monto asegurado directo en CLP
--------------------------------------------------------------*/

SELECT
    -- Identificación de la compañía aseguradora
    99017000 AS RUT_COMPANIA,
    2 AS DV_RUT_COMPANIA,
    'Seguros Suramericana S.A.' AS NOMBRE_COMPANIA,

    -- Datos consolidados
    a.*,

    -- RUT del contratante (valor fijo)
    70016160 AS RUT_CONTRATANTE,
    9 AS DV_CONTRANTE,

    /*----------------------------------------------------------
      Determinación de vigencia:
      - Documentos especiales: vigentes si último mes = vigencia_especial
      - Resto: vigentes si último mes = vigencia_general
    ----------------------------------------------------------*/
    CASE
        WHEN nnumdocu IN ({polizas_sql})
             AND max_mes_referencia = {vigencia_especial} THEN 1
        WHEN nnumdocu NOT IN ({polizas_sql})
             AND max_mes_referencia = {vigencia_general} THEN 1
        ELSE 0
    END AS vigentes,

    /*----------------------------------------------------------
      Cálculo de monto asegurado directo en CLP
      - Depende del plan técnico
      - Se multiplica ValorCuota por un factor fijo
    ----------------------------------------------------------*/
    CASE
        WHEN a.PLAN_TECNICO = 4331 THEN b.ValorCuota * 3
        WHEN a.PLAN_TECNICO = 4332 THEN b.ValorCuota * 6
        WHEN a.PLAN_TECNICO = 4333 THEN b.ValorCuota * 8
        WHEN a.PLAN_TECNICO = 4334 THEN b.ValorCuota * 12
        ELSE b.ValorCuota * 4
    END AS MONTO_ASEGURADO_DIRECTO_CLP,

    -- Subdivisión informada a CMF
    '3B' AS Subdiv,

    -- Código de ramo CMF
    33 AS RAMO_CMF
INTO {oficio_table}
FROM #Oficio_LA_Araucana_PPI2_BKP a
LEFT JOIN {T["total_araucana"]} b
    -- Se cruza por mes de recaudación
    ON a.max_mes_referencia = b.MES_Recaudacion
   -- Ítem / folio de crédito
   AND a.NNUMITEM = b.foliocredito
   -- Número de póliza
   AND a.nnumdocu = b.POLIZA;
"""

fun.exec_sql(query_monitoreo_17, connection_string)

📦 Bloque 19 — Consulta LINKED SERVER (control)

In [ ]:
query_monitoreo_18 = """
/*--------------------------------------------------------------
 5. CONSULTA A BASE EXTERNA (LINKED SERVER)
    - Permite validar información de documentos específicos
    - Uso informativo / control
--------------------------------------------------------------*/

SELECT *
FROM OPENQUERY (
    SOL,
    'SELECT *
     FROM wrkroyal.ofi113857
     WHERE DOCUMENTO IN (
        4668266, 4780716, 4780717, 4780718, 4780719,
        5698774, 6354562, 7561166, 4659577, 4780715
     )'
);
"""

fun.exec_sql(query_monitoreo_18, connection_string)

📦 Bloque 20 — Control final: vigentes vs no vigentes

In [ ]:
query_monitoreo_19 = f"""
/*--------------------------------------------------------------
 6. CONTROL FINAL — RESUMEN DE REGISTROS
    - Cuenta pólizas vigentes vs no vigentes
    - Validación previa al envío CMF
--------------------------------------------------------------*/

SELECT
    VIGENTES,
    COUNT(NNUMDOCU) AS REGISTROS
FROM {oficio_table}
GROUP BY VIGENTES;
"""

df_control = fun.query_to_df(
    sql=query_monitoreo_19,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_control)